# Multi-Theme Sentiment Alpha from Search API

Build a dataset of theme-weighted sentiment scores using semantic search through Search API, then uploading it to WorldQuant BRAIN for alpha simulation.

**Why multi-theme?**
- Single-theme signals might be noisy; using orthogonal themes increases robustness
- Each theme captures a distinct dimension of company-level news sentiment
- Direction-aware weighting (positive vs negative themes) produces a balanced composite signal

**Workflow overview:**
**Universe & Theme Definition** — load entity universe, define 5 financial themes
**Volume Pre-Scan** — estimate data availability per theme using Volume API
**Search Execution** — execute optimized batch searches across themes × months × entities
**Entity Attribution & Scoring** — attribute search results to entities, compute weighted sentiment
**Matrix Construction** — build wide matrices (date × entity) for each signal
**QA Validation** — comprehensive data quality checks
**Upload to BRAIN** — upload MATRIX data fields for alpha simulation

  **Create a `.env` file with:**

  **WorldQuant Brain (required):**
  ```BRAIN_EMAIL=your_email```
  ```BRAIN_PASSWORD=your_password```

  **Note:** Restart the kernel if you change API credentials, as they are read at import time.


In [1]:
import asyncio
import requests
import json
import os
import io
import time
import re
import hashlib
import logging
import threading
import pickle
from collections import defaultdict, deque
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple, Set, Any
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    datefmt='%H:%M:%S'
)
log = logging.getLogger("thematic_alpha")

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

OUTPUT_DIR = "output_thematic"
CACHE_DIR = os.path.join(OUTPUT_DIR, "cache")
SEARCH_CACHE_DIR = os.path.join(OUTPUT_DIR, "search_cache")
CSV_DIR = os.path.join(OUTPUT_DIR, "csv")
PARQUET_DIR = os.path.join(OUTPUT_DIR, "parquet")

for d in [CACHE_DIR, SEARCH_CACHE_DIR, CSV_DIR, PARQUET_DIR]:
    os.makedirs(d, exist_ok=True)

In [2]:
from session import BrainSession

API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")

SEARCH_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/search"
VOLUME_ENDPOINT = f"{SEARCH_ENDPOINT}/volume"
COMENTIONS_ENDPOINT = f"{SEARCH_ENDPOINT}/co-mentions/entities"
KG_ENTITIES_ENDPOINT = f"{API_BASE_URL}/bigdata/v1/knowledge-graph/entities/id"
UPLOAD_ENDPOINT = f"{API_BASE_URL}/users/self/data-fields"


session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

USER_ID = session.get("/users/self").json()["id"]

class RateLimiter:
    
    def __init__(self, max_requests_per_minute: int = 30):
        self.max_rpm = max_requests_per_minute
        self.timestamps = deque()
        self.lock = threading.Lock()

    def wait(self):
        with self.lock:
            now = time.time()
            while self.timestamps and self.timestamps[0] < now - 60:
                self.timestamps.popleft()
            if len(self.timestamps) >= self.max_rpm:
                sleep_time = self.timestamps[0] + 60 - now + 0.1
                log.debug(f"Rate limit: sleeping {sleep_time:.1f}s")
                time.sleep(sleep_time)
            self.timestamps.append(time.time())

rate_limiter = RateLimiter(max_requests_per_minute=30)

def api_request(endpoint: str, payload: dict, max_retries: int = 5) -> Optional[dict]:

    for attempt in range(max_retries):
        rate_limiter.wait()
        try:
            response = session.post(endpoint, json=payload, timeout=60)

            if response.status_code == 200:
                return response.json()
            elif response.status_code == 429:
                wait = int(response.headers.get('Retry-After', 2 ** (attempt + 2)))
                log.warning(f"429 Rate Limited. Waiting {wait}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
            elif response.status_code in (403, 503):
                wait = 60 * (attempt + 1)
                log.error(f"{response.status_code} Blocked. Waiting {wait}s")
                time.sleep(wait)
            else:
                log.error(f"HTTP {response.status_code}: {response.text[:300]}")
                time.sleep(2 ** (attempt + 1))
        except requests.exceptions.Timeout:
            log.warning(f"Timeout (attempt {attempt+1}/{max_retries})")
            time.sleep(2 ** (attempt + 1))
        except Exception as e:
            log.error(f"Error: {e} (attempt {attempt+1}/{max_retries})")
            time.sleep(2 ** (attempt + 1))

    return None


Biometrics authentication required. Open this URL in your browser and complete 2FA:
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_bMvVR9cTKXPwHmTRdit68d9UQQQW


In [3]:
UNIVERSE_FILE = 'universe.json'
with open(UNIVERSE_FILE, 'r') as f:
    top3000 = json.load(f)

valid_ids = set(top3000)

print(f"Total entity IDs: {len(top3000)}")
print(f"Sample: {top3000[:5]}")


Total entity IDs: 3433
Sample: ['00067A', '001F1B', '002A99', '003B70', '006697']


In [4]:
START_DATE = "2021-01-01"
END_DATE = "2022-12-31"

def monthly_buckets(start, end):
    buckets, current = [], datetime.strptime(start, "%Y-%m-%d")
    final = datetime.strptime(end, "%Y-%m-%d")
    while current <= final:
        ms = current.replace(day=1)
        me = (ms.replace(month=ms.month % 12 + 1, day=1) if ms.month < 12
              else ms.replace(year=ms.year + 1, month=1, day=1)) - timedelta(days=1)
        if me > final: me = final
        buckets.append((ms.strftime("%Y-%m-%d"), me.strftime("%Y-%m-%d")))
        current = me + timedelta(days=1)
    return buckets

MONTHS = monthly_buckets(START_DATE, END_DATE)


## Theme Definition with Sentiment Filters

We use the Search API's `sentiment` filter to retrieve ONLY 
positive or negative chunks at the API level. This may produce purer signals than 
fetching all sentiment and filtering.


In [5]:
THEMES = {
    "earnings_positive": {
        "text": "the company reports strong earnings growth revenue beat expectations profit margins expanding",
        "description": "Clearly positive earnings news only",
        "sentiment_filter": {"min": 0.3, "max": 1.0},
        "direction": 1.0,
    },
    "supply_chain_negative": {
        "text": "supply chain disruption shortage delays logistics bottleneck inventory constraints",
        "description": "Clearly negative supply chain news only",
        "sentiment_filter": {"min": -1.0, "max": -0.3},
        "direction": -1.0,
    },
}

BATCH_SIZE = 500
entity_batches = [top3000[i:i+BATCH_SIZE] for i in range(0, len(top3000), BATCH_SIZE)]
n_batches = len(entity_batches)
total_q = len(THEMES) * len(MONTHS) * n_batches

for name, cfg in THEMES.items():
    print(f"  [{name}]: sentiment={cfg['sentiment_filter']}, dir={cfg['direction']:+.0f}")
print(f"\n{total_q} queries")

  [earnings_positive]: sentiment={'min': 0.3, 'max': 1.0}, dir=+1
  [supply_chain_negative]: sentiment={'min': -1.0, 'max': -0.3}, dir=-1

336 queries


## Search Execution

Execute sentiment-filtered semantic search. Each query uses the `sentiment` filter 
in the payload to restrict at the API level. All results cached to disk.

In [7]:

stats = defaultdict(lambda: {"api": 0, "cached": 0, "docs": 0, "chunks": 0})

for t_idx, (theme_name, cfg) in enumerate(THEMES.items()):
    print(f"\n{'─'*60}\n  Theme [{t_idx+1}/{len(THEMES)}]: {theme_name}\n{'─'*60}")

    for ms, me in MONTHS:
        m_docs = 0
        for b_idx, batch in enumerate(entity_batches):
            h = hashlib.md5(f"{theme_name}:{ms}:{me}:{b_idx}".encode()).hexdigest()[:12]
            cf = os.path.join(CACHE_DIR, f"{theme_name}_{ms[:7]}_{b_idx}_{h}.json")

            if os.path.exists(cf):
                stats[theme_name]["cached"] += 1
                with open(cf) as f:
                    d = json.load(f)
                    m_docs += len(d.get("results", []))
                continue

            payload = {
                "query": {
                    "text": cfg["text"],
                    "auto_enrich_filters": False,
                    "filters": {
                        "timestamp": {"start": f"{ms}T00:00:00Z", "end": f"{me}T23:59:59Z"},
                        "entity": {"any_of": batch},
                        "sentiment": cfg["sentiment_filter"],
                    },
                    "ranking_params": {"freshness_boost": 0},
                    "max_chunks": 1000,
                }
            }

            result = api_request(SEARCH_ENDPOINT, payload)
            stats[theme_name]["api"] += 1

            if result:
                cached = {"theme": theme_name, "batch_idx": b_idx,
                         "results": result.get("results", []),
                         "usage": result.get("usage", {})}
                with open(cf, 'w') as f:
                    json.dump(cached, f)
                docs = cached["results"]
                m_docs += len(docs)
                stats[theme_name]["docs"] += len(docs)
                stats[theme_name]["chunks"] += sum(len(d.get("chunks", [])) for d in docs)

        log.info(f"  {ms[:7]}: {m_docs} docs")

for tn, s in stats.items():
    print(f"  [{tn}]: {s['api']} API, {s['cached']} cached, {s['docs']} docs, {s['chunks']} chunks")

02:56:39 | INFO    |   2021-01: 1354 docs



────────────────────────────────────────────────────────────
  Theme [1/2]: earnings_positive
────────────────────────────────────────────────────────────


02:56:39 | INFO    |   2021-02: 1494 docs
02:56:39 | INFO    |   2021-03: 1141 docs
02:56:40 | INFO    |   2021-04: 1563 docs
02:56:40 | INFO    |   2021-05: 1527 docs
02:56:40 | INFO    |   2021-06: 1239 docs
02:56:41 | INFO    |   2021-07: 1636 docs
02:56:41 | INFO    |   2021-08: 1491 docs
02:56:41 | INFO    |   2021-09: 1219 docs
02:56:41 | INFO    |   2021-10: 1475 docs
02:56:42 | INFO    |   2021-11: 1320 docs
02:56:42 | INFO    |   2021-12: 1151 docs
02:56:42 | INFO    |   2022-01: 1272 docs
02:56:43 | INFO    |   2022-02: 1303 docs
02:56:43 | INFO    |   2022-03: 1067 docs
02:56:43 | INFO    |   2022-04: 1427 docs
02:56:43 | INFO    |   2022-05: 1320 docs
02:56:44 | INFO    |   2022-06: 1014 docs
02:56:44 | INFO    |   2022-07: 1375 docs
02:56:44 | INFO    |   2022-08: 1298 docs
02:56:44 | INFO    |   2022-09: 918 docs
02:56:45 | INFO    |   2022-10: 1507 docs
02:56:45 | INFO    |   2022-11: 1342 docs
02:56:45 | INFO    |   2022-12: 1180 docs
02:56:45 | INFO    |   2021-01: 103


────────────────────────────────────────────────────────────
  Theme [2/2]: supply_chain_negative
────────────────────────────────────────────────────────────


02:56:46 | INFO    |   2021-02: 249 docs
02:56:46 | INFO    |   2021-03: 326 docs
02:56:46 | INFO    |   2021-04: 211 docs
02:56:46 | INFO    |   2021-05: 245 docs
02:56:46 | INFO    |   2021-06: 227 docs
02:56:47 | INFO    |   2021-07: 277 docs
02:56:47 | INFO    |   2021-08: 321 docs
02:56:47 | INFO    |   2021-09: 494 docs
02:56:47 | INFO    |   2021-10: 667 docs
02:56:48 | INFO    |   2021-11: 700 docs
02:56:48 | INFO    |   2021-12: 331 docs
02:56:48 | INFO    |   2022-01: 313 docs
02:56:48 | INFO    |   2022-02: 461 docs
02:56:48 | INFO    |   2022-03: 354 docs
02:56:49 | INFO    |   2022-04: 334 docs
02:56:49 | INFO    |   2022-05: 373 docs
02:56:49 | INFO    |   2022-06: 267 docs
02:56:49 | INFO    |   2022-07: 291 docs
02:56:50 | INFO    |   2022-08: 350 docs
02:56:50 | INFO    |   2022-09: 210 docs
02:56:50 | INFO    |   2022-10: 246 docs
02:56:50 | INFO    |   2022-11: 333 docs
02:56:50 | INFO    |   2022-12: 236 docs


  [earnings_positive]: 0 API, 168 cached, 0 docs, 0 chunks
  [supply_chain_negative]: 0 API, 168 cached, 0 docs, 0 chunks


## Entity Attribution & Scoring

Attribute search results back to queried entities. Then compute relevance-weighted 
sentiment per entity per day.

**Attribution logic:**
- **Direct:** Entity detected in chunk AND present in queried batch → full score
- **Co-mention:** Entity in universe but not queried batch → 0.5× weight
- **Dropped:** No universe entity found in chunk detections

In [8]:

batches_as_sets = [set(b) for b in entity_batches]

theme_data = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

files = sorted(f for f in os.listdir(CACHE_DIR) if f.endswith(".json"))

attr_stats = defaultdict(lambda: {"direct": 0, "comention": 0, "none": 0})

for fname in files:
    with open(os.path.join(CACHE_DIR, fname)) as f:
        cached = json.load(f)

    tn = cached.get("theme", "unknown")
    bi = cached.get("batch_idx", 0)
    queried = batches_as_sets[bi] if bi < len(batches_as_sets) else valid_ids

    for doc in cached.get("results", []):
        ts = doc.get("timestamp", "")
        if not ts: continue
        date = ts[:10]

        doc_dets = set()
        scores = []
        for chunk in doc.get("chunks", []):
            rel = chunk.get("relevance", 0) or 0
            sent = chunk.get("sentiment", 0) or 0
            scores.append((rel, sent))
            for det in chunk.get("detections", []):
                if det.get("type") == "entity":
                    doc_dets.add(det.get("id", ""))

        if not scores: continue
        total_rel = sum(r for r, _ in scores)
        doc_ws = sum(r * s for r, s in scores) / total_rel if total_rel > 0 else 0
        avg_rel = np.mean([r for r, _ in scores])

        direct = queried & doc_dets
        if direct:
            for eid in direct:
                theme_data[tn][eid][date].append((avg_rel, doc_ws))
            attr_stats[tn]["direct"] += len(direct)
        else:
            universe_match = doc_dets & valid_ids
            if universe_match:
                for eid in universe_match:
                    theme_data[tn][eid][date].append((avg_rel * 0.5, doc_ws))
                attr_stats[tn]["comention"] += len(universe_match)
            else:
                attr_stats[tn]["none"] += 1

theme_dfs = {}
for tn, edict in theme_data.items():
    rows = []
    for eid, ddict in edict.items():
        for dt, pairs in ddict.items():
            tr = sum(r for r, _ in pairs)
            ws = sum(r * s for r, s in pairs) / tr if tr > 0 else 0
            rows.append({"date": dt, "entity_id": eid,
                        "weighted_sentiment": round(ws, 6),
                        "n_mentions": len(pairs)})
    if rows:
        df = pd.DataFrame(rows)
        df["date"] = pd.to_datetime(df["date"])
        theme_dfs[tn] = df.sort_values(["date", "entity_id"]).reset_index(drop=True)
        log.info(f"  [{tn}]: {len(df)} rows, {df['entity_id'].nunique()} entities, {df['date'].nunique()} dates")

print(f"\nAttribution:")
for tn, s in attr_stats.items():
    total = s['direct'] + s['comention'] + s['none']
    pct = s['direct'] / max(total, 1) * 100
    print(f"  [{tn}]: {pct:.0f}% direct ({s['direct']}/{total})")

02:57:00 | INFO    |   [earnings_positive]: 14901 rows, 2116 entities, 718 dates
02:57:00 | INFO    |   [supply_chain_negative]: 3108 rows, 639 entities, 608 dates



Attribution:
  [earnings_positive]: 69% direct (23136/33410)
  [supply_chain_negative]: 51% direct (4168/8166)


## Rolling Signal Construction

Instead of raw daily scores, we apply 
backward-looking rolling windows to smooth the signal:
- **7-day rolling** — responsive, captures short-term shifts
- **30-day rolling** — stable, captures trend

In [9]:

all_dates = pd.date_range(START_DATE, END_DATE, freq='D')
all_date_strs = all_dates.strftime('%Y-%m-%d').tolist()
entity_list = list(top3000)


def build_rolling_matrices(long_df: pd.DataFrame, 
                           windows: List[int] = [7, 30]) -> Dict[str, pd.DataFrame]:

    if long_df.empty:
        empty = pd.DataFrame(0.0, index=all_date_strs, columns=entity_list)
        return {"raw": empty, "7d": empty, "30d": empty}
    
    # Pivot to wide
    df = long_df.copy()
    df['date_str'] = df['date'].dt.strftime('%Y-%m-%d')
    
    pivot = df.pivot_table(
        index='date_str', columns='entity_id',
        values='weighted_sentiment', aggfunc='mean'
    )
    
    pivot = pivot.reindex(index=all_date_strs, columns=entity_list, fill_value=np.nan)
    
    result = {"raw": pivot.fillna(0.0).copy()}
    
    for w in windows:

        rolled = pivot.rolling(window=w, min_periods=1).mean()
        rolled = rolled.fillna(0.0)
        result[f"{w}d"] = rolled
    
    return result

theme_rolling = {}

for theme_name, long_df in theme_dfs.items():
    print(f"\n  [{theme_name}]")
    
    matrices = build_rolling_matrices(long_df, windows=[7, 30])
    theme_rolling[theme_name] = matrices
    
    for label, mat in matrices.items():
        nonzero = (mat != 0).sum().sum()
        entities_active = (mat != 0).any().sum()
        print(f"    {label:>4}: {nonzero:,} non-zero cells, {entities_active} entities")

composite_signals = {}

for window_label in ["raw", "7d", "30d"]:
    composite = pd.DataFrame(0.0, index=all_date_strs, columns=entity_list)
    
    for theme_name, cfg in THEMES.items():
        direction = cfg["direction"]
        mat = theme_rolling[theme_name][window_label]
     
        composite += mat * direction * 0.5
    
    for date_idx in composite.index:
        row = composite.loc[date_idx]
        mask = row != 0
        if mask.sum() > 1:
            composite.loc[date_idx, mask] = row[mask] - row[mask].mean()
    
    composite_signals[window_label] = composite
    
    nz = (composite != 0).sum().sum()
    ents = (composite != 0).any().sum()
    vals = composite.values[composite.values != 0]
    mean_v = np.mean(vals) if len(vals) > 0 else 0
    print(f"  composite_{window_label}: {nz:,} non-zero, {ents} entities, mean={mean_v:+.4f}")


  [earnings_positive]
     raw: 14,882 non-zero cells, 2114 entities
      7d: 81,883 non-zero cells, 2114 entities
     30d: 277,916 non-zero cells, 2114 entities

  [supply_chain_negative]
     raw: 3,096 non-zero cells, 635 entities
      7d: 17,429 non-zero cells, 635 entities
     30d: 57,521 non-zero cells, 635 entities
  composite_raw: 17,543 non-zero, 2186 entities, mean=+0.0002
  composite_7d: 94,560 non-zero, 2186 entities, mean=+0.0000
  composite_30d: 309,177 non-zero, 2186 entities, mean=+0.0000


## Data QA Validation

Trying to define tests for quick quality checks before uploading to BRAIN.

In [10]:
qa_pass, qa_warn, qa_fail = 0, 0, 0

print("\nQA-1: Date Range")
for label, mat in composite_signals.items():
    ok = len(mat) == len(all_date_strs) and mat.index[0] == START_DATE and mat.index[-1] == END_DATE
    if ok: qa_pass += 1
    else: qa_fail += 1
    print(f" composite_{label}: {len(mat)} days, {mat.index[0]} to {mat.index[-1]}")

print("\nQA-2: Entity Coverage")
for theme_name, matrices in theme_rolling.items():
    raw = matrices["raw"]
    ents = (raw != 0).any().sum()
    pct = ents / len(entity_list) * 100
    if pct > 5: qa_pass += 1
    elif pct > 1: qa_warn += 1
    else: qa_fail += 1
    print(f"[{theme_name}]: {ents}/{len(entity_list)} entities ({pct:.1f}%)")

print("\nQA-3: No NaN/Inf")
for label, mat in composite_signals.items():
    has_nan = mat.isna().any().any()
    has_inf = np.isinf(mat.values).any()
    ok = not has_nan and not has_inf
    if ok: qa_pass += 1
    else: qa_fail += 1
    print(f" composite_{label}: NaN={has_nan}, Inf={has_inf}")

print("\nQA-4: Value Range")
for theme_name, matrices in theme_rolling.items():
    for label, mat in matrices.items():
        vals = mat.values[mat.values != 0]
        if len(vals) == 0: continue
        mn, mx = vals.min(), vals.max()
        ok = mn >= -1.5 and mx <= 1.5  
        if ok: qa_pass += 1
        else: qa_warn += 1
        print(f" [{theme_name}_{label}]: [{mn:+.3f}, {mx:+.3f}]")

print("\n QA-5: Composite Balance")
for label, mat in composite_signals.items():
    vals = mat.values[mat.values != 0]
    if len(vals) == 0: continue
    n_pos = (vals > 0).sum()
    pct_pos = n_pos / len(vals) * 100
    mean_v = np.mean(vals)
    ok_balance = 20 < pct_pos < 80
    ok_mean = abs(mean_v) < 0.1
    if ok_balance and ok_mean: qa_pass += 1
    else: qa_warn += 1
    print(f" composite_{label}: {pct_pos:.0f}% positive, mean={mean_v:+.4f}")

print("\nQA-6: Rolling Smoothing (30d should be smoother than raw)")
for theme_name, matrices in theme_rolling.items():
    raw_std = matrices["raw"].values[matrices["raw"].values != 0].std() if (matrices["raw"] != 0).any().any() else 0
    d30_std = matrices["30d"].values[matrices["30d"].values != 0].std() if (matrices["30d"] != 0).any().any() else 0
    ok = d30_std <= raw_std or raw_std == 0
    if ok: qa_pass += 1
    else: qa_warn += 1
    print(f" [{theme_name}]: raw_std={raw_std:.4f}, 30d_std={d30_std:.4f}")

total = qa_pass + qa_warn + qa_fail

print(f"Passed: {qa_pass} Warnings: {qa_warn} Failed: {qa_fail}")



QA-1: Date Range
 composite_raw: 730 days, 2021-01-01 to 2022-12-31
 composite_7d: 730 days, 2021-01-01 to 2022-12-31
 composite_30d: 730 days, 2021-01-01 to 2022-12-31

QA-2: Entity Coverage
[earnings_positive]: 2114/3433 entities (61.6%)
[supply_chain_negative]: 635/3433 entities (18.5%)

QA-3: No NaN/Inf
 composite_raw: NaN=False, Inf=False
 composite_7d: NaN=False, Inf=False
 composite_30d: NaN=False, Inf=False

QA-4: Value Range
 [earnings_positive_raw]: [-0.800, +0.880]
 [earnings_positive_7d]: [-0.800, +0.880]
 [earnings_positive_30d]: [-0.780, +0.870]
 [supply_chain_negative_raw]: [-0.890, +0.770]
 [supply_chain_negative_7d]: [-0.890, +0.770]
 [supply_chain_negative_30d]: [-0.890, +0.770]

 QA-5: Composite Balance
 composite_raw: 63% positive, mean=+0.0002
 composite_7d: 63% positive, mean=+0.0000
 composite_30d: 61% positive, mean=+0.0000

QA-6: Rolling Smoothing (30d should be smoother than raw)
 [earnings_positive]: raw_std=0.2835, 30d_std=0.2681
 [supply_chain_negative]: r

## Upload to BRAIN

Let's try uploading these 4 data fields:
1. `{USER_ID}_earnings_pos_7d` — 7-day rolling positive earnings sentiment
2. `{USER_ID}_earnings_pos_30d` — 30-day rolling positive earnings sentiment
3. `{USER_ID}_supply_neg_7d` — 7-day rolling negative supply chain sentiment
4. `{USER_ID}_supply_neg_30d` — 30-day rolling negative supply chain sentiment


In [11]:
def upload_matrix(field_id: str, matrix_df: pd.DataFrame, 
                  scale: float = 100.0, source: str = "BigData",
                  max_retries: int = 3, retry_sleep: int = 30) -> bool:

    df = matrix_df.copy() * scale
    df = df.astype(float)
    df.index = pd.to_datetime(df.index)
    df.index.name = 'date'

    buf = io.BytesIO()
    table = pa.Table.from_pandas(df, preserve_index=True)
    pq.write_table(table, buf, compression='GZIP')
    parquet_bytes = buf.getvalue()

    local = os.path.join(PARQUET_DIR, f"{field_id}.parquet")
    with open(local, 'wb') as f:
        f.write(parquet_bytes)

    size_kb = len(parquet_bytes) / 1024

    field_meta = json.dumps({
        'id': field_id, 'type': 'MATRIX', 'source': source,
        'data': [{'region': 'USA', 'delay': 1}]
    })

    for attempt in range(1, max_retries + 1):
        try:
            upload_buf = io.BytesIO(parquet_bytes)
            r = session.post(
                UPLOAD_ENDPOINT,
                data={'field': field_meta},
                files={'data': upload_buf},
                timeout=120
            )
            if r.status_code in [200, 201]:
                log.info(f"{field_id} ({size_kb:.0f} KB)")
                return True
            elif r.status_code == 409:
                log.warning(f" {field_id} already exists")
                return True
            else:
                log.error(f"{field_id}: HTTP {r.status_code} — {r.text[:200]}")
                if attempt < max_retries:
                    wait = retry_sleep * attempt
                    log.info(f"     Retrying in {wait}s (attempt {attempt}/{max_retries})...")
                    time.sleep(wait)
                    continue
                return False
        except Exception as e:
            log.error(f"{field_id}: {e}")
            if attempt < max_retries:
                wait = retry_sleep * attempt
                log.info(f"Retrying in {wait}s (attempt {attempt}/{max_retries})...")
                time.sleep(wait)
                continue
            return False

    return False

uploads = {
    f"{USER_ID}_earnings_pos_7d": theme_rolling["earnings_positive"]["7d"],
    f"{USER_ID}_earnings_pos_30d": theme_rolling["earnings_positive"]["30d"],
    f"{USER_ID}_supply_neg_7d": theme_rolling["supply_chain_negative"]["7d"],
    f"{USER_ID}_supply_neg_30d": theme_rolling["supply_chain_negative"]["30d"],
}

results = {}
for field_id, matrix in uploads.items():
    nz = (matrix != 0).sum().sum()
    ents = (matrix != 0).any().sum()
    print(f"\n  [{field_id}]")
    print(f"    Non-zero: {nz:,} | Entities: {ents}")
    results[field_id] = upload_matrix(field_id, matrix, scale=100.0)



  [AS37074_earnings_pos_7d]
    Non-zero: 81,883 | Entities: 2114


02:57:22 | INFO    | AS37074_earnings_pos_7d (2145 KB)



  [AS37074_earnings_pos_30d]
    Non-zero: 277,916 | Entities: 2114


02:57:26 | INFO    | AS37074_earnings_pos_30d (2161 KB)



  [AS37074_supply_neg_7d]
    Non-zero: 17,429 | Entities: 635


02:57:30 | INFO    | AS37074_supply_neg_7d (2007 KB)



  [AS37074_supply_neg_30d]
    Non-zero: 57,521 | Entities: 635


02:57:34 | INFO    | AS37074_supply_neg_30d (2012 KB)
